# ODI to Databricks Migration: `insert_hr_trg_dep.sql`

**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook contains the converted Spark SQL for loading data into the `TRG_DEP` table from the `DEPARTMENTS` table.
The original ODI `SCEN_TASK_NO` blocks are represented by distinct cells, and Oracle-specific syntax has been migrated to Databricks Spark SQL equivalents.

In [ ]:
import datetime

# Create placeholder widgets as per standard notebook structure, even if not directly used in this specific SQL task.
# These widgets allow for external parameterization of the notebook when executed as a job.
dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type (e.g., FULL, INCREMENTAL)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "2. Data Source Numeric ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "3. ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "4. ODI Session Number")

# You can also add more specific widgets if needed for this particular process, e.g., for date filtering.
# Example: dbutils.widgets.text("LAST_EXTRACT_DATE", (datetime.date.today() - datetime.timedelta(days=1)).strftime("%Y-%m-%d"), "Last Extract Date (YYYY-MM-DD)")
# Example: dbutils.widgets.text("CURRENT_EXTRACT_DATE", datetime.date.today().strftime("%Y-%m-%d"), "Current Extract Date (YYYY-MM-DD)")

## ETL Parameters

The following parameters are used to control the ETL process. Their values are set via widgets or defaults.

In [ ]:
# Display current widget values for review
print(f"ETL_JOB_TYPE: {dbutils.widgets.get('ETL_JOB_TYPE')}")
print(f"DATASOURCE_NUM_ID: {dbutils.widgets.get('DATASOURCE_NUM_ID')}")
print(f"ETL_PROC_WID: {dbutils.widgets.get('ETL_PROC_WID')}")
print(f"ODI_SESS_NO: {dbutils.widgets.get('ODI_SESS_NO')}")

display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID,
    ${ETL_PROC_WID} AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

## Insert into Target Table (`TRG_DEP`)

**SCEN_TASK_NO {10, 20, 30}**: This step corresponds to the data insertion task from the source `DEPARTMENTS` table into the target `TRG_DEP` table.

Oracle-specific hints `/*+ APPEND PARALLEL */` have been removed as they are not applicable to Delta Lake in Spark SQL. Schema and table names have been converted to the `workspace.<schema_lower>.<table_lower>` format.

In [ ]:
%sql
INSERT INTO workspace.hr.trg_dep
  (
    department_id,
    department_name,
    manager_id,
    location_id
  )
SELECT
  departments.department_id,
  departments.department_name,
  departments.manager_id,
  departments.location_id
FROM
  workspace.hr.departments AS departments;

## Validation

Review the number of records in the target table after the insert operation.

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_dep
FROM workspace.hr.trg_dep;

In [ ]:
%sql
SELECT *
FROM workspace.hr.trg_dep
ORDER BY department_id
LIMIT 10;

## Conversion Notes

*   **Oracle Hints Removed:** `/*+ APPEND PARALLEL */` was removed from the INSERT statement as it's not applicable in Spark SQL/Delta Lake.
*   **Schema and Table Naming:** All schema and table references (`HR.TRG_DEP`, `HR.DEPARTMENTS`) have been converted to `workspace.hr.trg_dep` and `workspace.hr.departments` respectively, following the `workspace.<schema_lower>.<table_lower>` convention.
*   **Column Naming:** Column names in the INSERT and SELECT clauses have been converted to lowercase for consistency, although Spark SQL handles case-insensitivity for unquoted identifiers.
*   **Task Grouping:** `SCEN_TASK_NO {10, 20, 30}` appear to be sequential markers for the same logical operation (the INSERT). They are grouped into a single relevant SQL cell with comments.
*   **No ETL Parameters Found:** The source SQL did not contain any explicit ODI `#GLOBAL` or `#SCHEMA` parameters. Standard placeholder widgets are created but not directly used in the converted SQL, only for display and consistency.